# VGG Cosmology Bias Check

This notebook summarizes the VGG-based cosmology encoder and uses it to check whether conditional HI diffusion samples preserve the requested cosmology.

The poster-level message is simple:

```text
requested cosmology -> conditional diffusion -> generated HI field -> VGG encoder -> recovered cosmology
```

If the recovered parameter follows the requested parameter, then the generated field carries the conditioning information. The cleanest current result is for `Omega_m`, so the poster figure below focuses on that one parameter.

In [ ]:
from pathlib import Path
import json
import math

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown, Image

# Find repository root whether this notebook is run from notebooks/ or from repo root.
ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / '.git').exists():
    ROOT = ROOT.parent

RESULT_ROOT = ROOT / 'results' / 'nf_conditional_bias_probe'
ENCODER_DIR = RESULT_ROOT / 'encoder'
CALIB_DIR = RESULT_ROOT / 'calibration_vgg'
POSTER_DIR = CALIB_DIR / 'poster'
POSTER_DIR.mkdir(parents=True, exist_ok=True)
POSTER_FIGS_DIR = ROOT / 'poster' / 'figs'
POSTER_FIGS_DIR.mkdir(parents=True, exist_ok=True)

BEST_VGG_ENCODER = ENCODER_DIR / 'vgg_mlp_big_avgmax.npz'
BEST_VGG_MODEL = ENCODER_DIR / 'vgg_mlp_big_avgmax.pkl'

REAL_METRICS_PATH = ENCODER_DIR / 'vgg_real_test_metrics.csv'
REAL_PRED_PATH = ENCODER_DIR / 'vgg_real_test_per_cosmology_predictions.csv'
VGG_COMPARISON_PATH = ENCODER_DIR / 'vgg_encoder_r2_comparison.csv'
CALIB_POINTS_PATH = CALIB_DIR / 'bias_probe_per_cosmology_points.csv'
CALIB_SLOPES_PATH = CALIB_DIR / 'bias_probe_regime_slopes.csv'
CALIB_METADATA_PATH = CALIB_DIR / 'bias_probe_eval_metadata.json'

PARAM_ORDER = ['Omega_m', 'sigma_8', 'A_SN1', 'A_AGN1', 'A_SN2', 'A_AGN2']
PARAM_LABEL = {
    'Omega_m': r'$\Omega_m$',
    'sigma_8': r'$\sigma_8$',
    'A_SN1': r'$A_{SN1}$',
    'A_AGN1': r'$A_{AGN1}$',
    'A_SN2': r'$A_{SN2}$',
    'A_AGN2': r'$A_{AGN2}$',
}
REGIME_LABEL = {
    'memorization': r'Memorization ($N_{2D}=128$)',
    'generalization': r'Generalization ($N_{2D}=16{,}384$)',
}
REGIME_COLOR = {
    'memorization': '#D55E00',    # Okabe-Ito vermillion
    'generalization': '#0072B2',  # Okabe-Ito blue
}
REGIME_MARKER = {'memorization': 'o', 'generalization': 's'}

plt.rcParams.update({
    'font.family': 'DejaVu Serif',
    'mathtext.fontset': 'dejavuserif',
    'axes.linewidth': 1.25,
    'xtick.major.width': 1.15,
    'ytick.major.width': 1.15,
    'xtick.major.size': 5.5,
    'ytick.major.size': 5.5,
})

def read_csv_or_none(path: Path) -> pd.DataFrame | None:
    if path.exists():
        return pd.read_csv(path)
    return None

def read_json_or_none(path: Path) -> dict:
    if path.exists():
        return json.loads(path.read_text())
    return {}

def show_missing(paths):
    missing = [str(p.relative_to(ROOT)) for p in paths if not p.exists()]
    if missing:
        display(Markdown('**Missing result files in this checkout:**\n\n' + '\n'.join(f'- `{p}`' for p in missing)))
    else:
        display(Markdown('All expected result files are present.'))

print('repo root:', ROOT)
show_missing([REAL_METRICS_PATH, REAL_PRED_PATH, CALIB_POINTS_PATH, CALIB_SLOPES_PATH])

## What The VGG Encoder Does

The encoder is only a diagnostic probe. It is not part of the diffusion model training.

```text
normalized HI slice
-> copy the one HI channel into R, G, B
-> bilinear resize to 224 x 224
-> frozen ImageNet VGG16 convolutional features
-> average + max pooling
-> MLP regression head with hidden layers 1024, 512, 256
-> predicted CAMELS parameters
```

Important safeguards:

- VGG16 is frozen; only the regression head is trained.
- The head is trained on real HI slices from non-held-out simulations only.
- Held-out simulations `900-931` are used for testing and for the diffusion calibration plot.
- The current best encoder is `frozen VGG16 + avg+max pooling + MLP(1024,512,256)`.

In [ ]:
# Recorded summary from the completed Great Lakes VGG tests.
# The live CSVs are preferred when present; this table is here so the notebook remains readable off-cluster.
recorded_vgg_results = pd.DataFrame([
    {'encoder': 'avg+max + MLP(1024,512,256), 65k slices', 'Omega_m': 0.9115, 'sigma_8': 0.7374, 'A_SN1': 0.4544, 'A_AGN1': -0.0063, 'A_SN2': 0.3251, 'A_AGN2': 0.1005},
    {'encoder': 'avg + MLP(1024,512,256), 65k slices',     'Omega_m': 0.9027, 'sigma_8': 0.7349, 'A_SN1': 0.4535, 'A_AGN1': -0.0313, 'A_SN2': 0.2561, 'A_AGN2': 0.0868},
    {'encoder': 'avg+max + MLP(512,256), 32k slices',      'Omega_m': 0.8992, 'sigma_8': 0.6830, 'A_SN1': 0.4189, 'A_AGN1': 0.0144, 'A_SN2': 0.2953, 'A_AGN2': 0.0946},
    {'encoder': 'avg+max + Ridge(alpha=1), 65k slices',    'Omega_m': 0.8977, 'sigma_8': 0.7143, 'A_SN1': 0.4145, 'A_AGN1': -0.0217, 'A_SN2': 0.2862, 'A_AGN2': 0.1418},
])

display(recorded_vgg_results)

## Real Held-Out Probe Validation

Before interpreting generated-map calibration, measure what the frozen VGG16+MLP probe can recover from
real held-out CAMELS simulations 900--931. For every parameter, the point is the median probe prediction
over real slices from one held-out cosmology; bars show the 16th--84th percentile slice spread.

The fitted **slope** measures response to that parameter, while $R^2$ measures absolute predictive agreement.
Both metrics are computed from the same held-out per-cosmology medians. Weak generated-map response is only
strong evidence of a generator failure when the real-map probe itself has a useful slope and $R^2$.


In [ ]:
REAL_METADATA_PATH = ENCODER_DIR / 'vgg_real_test_metadata.json'
FULL_SWEEP_METADATA_PATH = (
    ROOT / 'results' / 'nf_conditional_bias_fresh_full_sweep_200k'
    / 'calibration_vgg' / 'bias_probe_eval_metadata.json'
)

real_metrics = read_csv_or_none(REAL_METRICS_PATH)
real_pred = read_csv_or_none(REAL_PRED_PATH)
real_probe_metadata = read_json_or_none(REAL_METADATA_PATH)
full_sweep_metadata = read_json_or_none(FULL_SWEEP_METADATA_PATH)

real_encoder = str(real_probe_metadata.get('encoder_path', 'unknown encoder'))
sweep_encoder = str(full_sweep_metadata.get('encoder_path', 'unknown encoder'))
probe_label = f'probe provenance: {real_encoder}'

if real_encoder != 'unknown encoder' and sweep_encoder != 'unknown encoder':
    if Path(real_encoder).name != Path(sweep_encoder).name:
        display(Markdown(
            '**Probe provenance mismatch:** the saved held-out-real predictions came from '
            f'`{real_encoder}`, while the full generated-map sweep used `{sweep_encoder}`. '
            'The plots below remain valid for the named probe, but they must not be used as the '
            'generator sweep baseline until matching held-out-real predictions are available.'
        ))
    else:
        display(Markdown(f'**Probe provenance matched:** `{Path(real_encoder).name}`'))

if real_metrics is not None:
    real_summary = real_metrics[
        (real_metrics['split'] == 'test') & (real_metrics['grain'] == 'per_cosmology')
    ].copy()
    real_summary['parameter'] = pd.Categorical(real_summary['parameter'], PARAM_ORDER, ordered=True)
    display(real_summary.sort_values('parameter')[['parameter', 'n', 'mae', 'rmse', 'bias', 'r2']])
else:
    display(Markdown('Live `vgg_real_test_metrics.csv` is not present in this checkout.'))


In [ ]:
if real_pred is None:
    display(Markdown(
        'Cannot compute all-parameter held-out slopes and $R^2$ because '
        '`vgg_real_test_per_cosmology_predictions.csv` is missing.'
    ))
else:
    import sys
    scripts_dir = ROOT / 'scripts'
    if str(scripts_dir) not in sys.path:
        sys.path.insert(0, str(scripts_dir))
    from plot_nf_conditional_bias_vgg_probe_validation import write_probe_validation_outputs

    real_probe_panel = ENCODER_DIR / 'vgg_real_probe_all_parameters_1to1.png'
    real_probe_summary_figure = ENCODER_DIR / 'vgg_real_probe_slope_r2_summary.png'
    real_probe_summary_table = ENCODER_DIR / 'vgg_real_probe_slope_r2_summary.csv'
    real_probe_summary = write_probe_validation_outputs(
        real_pred,
        panel_path=real_probe_panel,
        summary_path=real_probe_summary_figure,
        table_path=real_probe_summary_table,
        probe_label=probe_label,
    )
    display(real_probe_summary)
    display(Image(filename=str(real_probe_panel)))
    display(Image(filename=str(real_probe_summary_figure)))
    print('wrote', real_probe_panel)
    print('wrote', real_probe_summary_figure)
    print('wrote', real_probe_summary_table)


### How to use this check

Do not compare generated slopes to one for a parameter the probe cannot read reliably from real maps.
For example, a strong real-map $Omega_m$ slope and $R^2$ make a weak generated $Omega_m$ response
diagnostic of the generator. Conversely, weak real-map performance for an astrophysical parameter means
the VGG probe is not a reliable verifier for that conditional direction.


## Generated-Field Calibration

For the diffusion test, the input is a held-out cosmology `theta_in`. For each held-out cosmology, the model generates multiple HI samples with different noise seeds but the same requested cosmology.

For one parameter, the plotted point is:

```text
median recovered theta over generated samples at fixed theta_in
```

The error bar is:

```text
16th to 84th percentile spread of recovered theta over those generated samples
```

The fitted slope is the headline number:

```text
slope near 0: generated fields mostly ignore the requested cosmology
slope near 1: generated fields track the requested cosmology
```

The poster figure below only shows `Omega_m`, where the signal is strongest and easiest to explain.

In [ ]:
points = read_csv_or_none(CALIB_POINTS_PATH)
slopes = read_csv_or_none(CALIB_SLOPES_PATH)
metadata = read_json_or_none(CALIB_METADATA_PATH)

if points is not None:
    display(points.head())
    display(points.groupby(['regime', 'dataset_size', 'parameter']).size().reset_index(name='n_points').head(12))
else:
    display(Markdown('Live generated-field calibration points are missing in this checkout.'))

if slopes is not None:
    slope_show = slopes[slopes['parameter'].isin(['Omega_m', 'sigma_8'])].copy()
    display(slope_show[['regime', 'dataset_size', 'parameter', 'slope', 'slope_ci16', 'slope_ci84', 'intercept', 'n_heldout']])

In [ ]:

def recovery_r2(truth, pred):
    truth = np.asarray(truth, dtype=float)
    pred = np.asarray(pred, dtype=float)
    denom = np.sum((truth - truth.mean()) ** 2)
    if denom <= 0:
        return np.nan
    return 1.0 - np.sum((pred - truth) ** 2) / denom


def plot_omega_calibration_poster(
    points: pd.DataFrame | None,
    slopes: pd.DataFrame | None,
    *,
    out_name='bias_probe_omega_m_best_vgg_poster.png',
    title=r'Generated HI fields recover $\Omega_m$',
):
    if points is None or slopes is None:
        display(Markdown('Cannot draw the poster calibration plot because the live calibration CSVs are missing.'))
        return None

    param = 'Omega_m'
    p = points[points['parameter'] == param].copy()
    if p.empty:
        display(Markdown(f'No rows for `{param}`.'))
        return None

    with plt.rc_context({
        'font.family': 'serif',
        'font.serif': ['DejaVu Serif'],
        'mathtext.fontset': 'dejavuserif',
        'axes.linewidth': 1.25,
        'xtick.major.width': 1.15,
        'ytick.major.width': 1.15,
    }):
        fig, ax = plt.subplots(figsize=(6.4, 6.15), constrained_layout=False)

        lo = float(min(p['theta_in'].min(), p['theta_rec_q16'].min()))
        hi = float(max(p['theta_in'].max(), p['theta_rec_q84'].max()))
        pad = 0.08 * max(hi - lo, 1e-6)
        lo -= pad
        hi += pad
        ideal_handle, = ax.plot(
            [lo, hi], [lo, hi], '--', color='0.25', lw=2.3,
            label='ideal recovery', zorder=1,
        )

        legend_handles = [ideal_handle]
        legend_labels = ['ideal recovery']
        r2_by_regime = {}
        slope_by_regime = {}

        for regime in ['memorization', 'generalization']:
            sub = p[p['regime'] == regime].sort_values('theta_in')
            if sub.empty:
                continue
            color = REGIME_COLOR[regime]
            y = sub['theta_rec_median'].to_numpy(float)
            yerr = np.vstack([
                np.maximum(y - sub['theta_rec_q16'].to_numpy(float), 0.0),
                np.maximum(sub['theta_rec_q84'].to_numpy(float) - y, 0.0),
            ])
            r2_by_regime[regime] = recovery_r2(sub['theta_in'].to_numpy(float), y)

            fit = slopes[(slopes['parameter'] == param) & (slopes['regime'] == regime)]
            slope = float(fit['slope'].iloc[0]) if len(fit) else np.nan
            intercept = float(fit['intercept'].iloc[0]) if len(fit) else np.nan
            slope_by_regime[regime] = slope

            n_label = r'$N_{2D}=2^7$' if regime == 'memorization' else r'$N_{2D}=2^{14}$'
            regime_label = 'mem. regime' if regime == 'memorization' else 'gen. regime'
            label = f'{regime_label} ({n_label})'
            if np.isfinite(slope):
                label += rf', slope={slope:.2f}'

            err = ax.errorbar(
                sub['theta_in'], y, yerr=yerr,
                fmt=REGIME_MARKER[regime], ms=7.4, lw=1.7, capsize=3.1, capthick=1.35,
                color=color, ecolor=color, alpha=0.92,
                markeredgecolor='white', markeredgewidth=0.75,
                label=label, zorder=3,
            )
            legend_handles.append(err)
            legend_labels.append(label)

            if np.isfinite(slope) and np.isfinite(intercept):
                xs = np.array([lo, hi])
                ax.plot(xs, slope * xs + intercept, color=color, lw=3.2, zorder=2)

        ax.set_xlim(lo, hi)
        ax.set_ylim(lo, hi)
        ax.set_xlabel(r'Truth $\Omega_m$', fontsize=18.0, labelpad=6)
        ax.set_ylabel(r'Prediction $\Omega_m$', fontsize=18.0, labelpad=8)
        ax.tick_params(labelsize=15.0, width=1.25, length=5.5)
        ax.set_aspect('equal', adjustable='box')
        ax.set_box_aspect(1)
        ax.grid(False)
        for spine in ('top', 'right'):
            ax.spines[spine].set_visible(False)
        for spine in ('left', 'bottom'):
            ax.spines[spine].set_linewidth(1.25)

        fig.suptitle(title, fontsize=20.5, y=0.985)
        fig.legend(
            legend_handles, legend_labels,
            loc='upper center', bbox_to_anchor=(0.5, 0.925),
            ncol=3, frameon=False, fontsize=10.8,
            handlelength=1.8, columnspacing=0.95,
            labelspacing=0.28, borderaxespad=0.0,
        )
        fig.subplots_adjust(left=0.135, right=0.985, bottom=0.115, top=0.81)

        out = POSTER_DIR / out_name
        fig.savefig(out, dpi=300, bbox_inches='tight')
        poster_out = POSTER_FIGS_DIR / out_name
        fig.savefig(poster_out, dpi=300, bbox_inches='tight')
        plt.show()
        print('wrote', out)
        print('wrote', poster_out)
        if r2_by_regime:
            print('Omega_m R^2 by regime:', {k: round(v, 3) for k, v in r2_by_regime.items()})
        return out

omega_plot = plot_omega_calibration_poster(points, slopes)


## One-Sentence Poster Caption

Using a frozen VGG16 feature encoder trained only on real non-held-out HI fields, the large-data conditional diffusion model recovers the requested `Omega_m` much better than the small-data model. This supports the interpretation that the generalization-regime model is not just producing plausible fields; it is more faithful to the input cosmology.

## Practical Interpretation

- `Omega_m` is the clearest result and the best poster panel.
- `sigma_8` has some signal but is weaker.
- Feedback parameters are much less reliable from HI alone with this encoder.
- The VGG encoder is a probe, not proof of perfect cosmological calibration. The result should be stated as evidence that conditioning fidelity improves in the higher-data regime.

In [ ]:
# Show the saved poster plot inside the notebook if it was created.
poster_fig_copy = POSTER_FIGS_DIR / 'bias_probe_omega_m_best_vgg_poster.png'
if 'omega_plot' in globals() and omega_plot is not None and Path(omega_plot).exists():
    display(Markdown(f'Saved notebook result: `{Path(omega_plot).relative_to(ROOT)}`'))
    display(Markdown(f'Saved poster copy: `{poster_fig_copy.relative_to(ROOT)}`'))
    display(Image(filename=str(omega_plot)))
elif poster_fig_copy.exists():
    display(Markdown(f'Live calibration CSVs are missing here, so showing the existing poster copy: `{poster_fig_copy.relative_to(ROOT)}`'))
    display(Image(filename=str(poster_fig_copy)))
else:
    display(Markdown('Poster plot not created in this environment because the live calibration CSVs are missing.'))


## Full training-size sweep

This section replaces the two-endpoint comparison with conditional UNet-128 models trained on every
$N_{2D}=2^6,\ldots,2^{15}$. In particular, all ten generators are trained from clean initializations;
no generator checkpoint from the earlier two-size comparison is reused. All models use the same 200k
optimizer-update target, fixed RNG seed, full six-dimensional CAMELS conditioning vector, heldout
simulations 900--931, and the same frozen VGG16+MLP probe.

The first figure overlays all ten fitted $\Omega_m$ responses and places their slopes directly against
training-set size. The second keeps the underlying per-size points and uncertainty intervals visible in
separate panels. The third summarizes the response slope for all six parameters. A slope of one is ideal;
a small slope means generated maps respond weakly to changes in the requested parameter.


In [ ]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import Image, Markdown, display

# Resolve the repository independently so this cell is safe to run by itself.
PROJECT_DIR = Path.cwd().resolve()
while PROJECT_DIR != PROJECT_DIR.parent and not (PROJECT_DIR / '.git').exists():
    PROJECT_DIR = PROJECT_DIR.parent
if not (PROJECT_DIR / '.git').exists():
    raise RuntimeError(f'Could not locate the repository root from {Path.cwd().resolve()}')

full_sweep_root = PROJECT_DIR / 'results' / 'nf_conditional_bias_fresh_full_sweep_200k' / 'calibration_vgg'
full_points_path = full_sweep_root / 'bias_probe_per_cosmology_points.csv'
full_slopes_path = full_sweep_root / 'bias_probe_regime_slopes.csv'
omega_figure = full_sweep_root / 'bias_probe_omega_m_all_dataset_sizes.png'
omega_transition_figure = full_sweep_root / 'bias_probe_omega_m_transition_vs_dataset_size.png'
slope_figure = full_sweep_root / 'bias_probe_all_parameter_slopes_vs_dataset_size.png'
expected_sizes = [2**power for power in range(6, 16)]

if full_points_path.exists() and full_slopes_path.exists():
    full_points = pd.read_csv(full_points_path)
    full_slopes = pd.read_csv(full_slopes_path)
    present_sizes = sorted(full_points['dataset_size'].astype(int).unique().tolist())
    missing_sizes = sorted(set(expected_sizes) - set(present_sizes))
    if missing_sizes:
        raise RuntimeError(f'missing dataset sizes: {missing_sizes}')
    display(pd.DataFrame({
        'dataset_size': expected_sizes,
        'log2_size': list(range(6, 16)),
        'point_rows': [int((full_points.dataset_size == size).sum()) for size in expected_sizes],
        'slope_rows': [int((full_slopes.dataset_size == size).sum()) for size in expected_sizes],
    }))
    scripts_dir = PROJECT_DIR / 'scripts'
    if str(scripts_dir) not in sys.path:
        sys.path.insert(0, str(scripts_dir))
    from plot_nf_conditional_bias_full_sweep import (
        plot_omega_m_grid,
        plot_omega_m_transition,
        plot_parameter_slope_summary,
    )
    plot_omega_m_grid(full_points, full_slopes, omega_figure)
    plot_omega_m_transition(full_points, full_slopes, omega_transition_figure)
    plot_parameter_slope_summary(full_slopes, slope_figure)
    display(Image(filename=str(omega_transition_figure)))
    display(Image(filename=str(omega_figure)))
    display(Image(filename=str(slope_figure)))
else:
    display(Markdown(
        '**Full-sweep results are not present yet.** Run the Great Lakes full-sweep pipeline, then rerun '
        'this cell. No sizes are interpolated or silently omitted.'
    ))


### Reading the sweep

Read the panels horizontally, not as ten independent anecdotes. The key question is whether the fitted
response moves smoothly toward the ideal line as training data grows, and whether different cosmological
and astrophysical parameters reach that regime at the same dataset size. The error bars describe variation
across generated samples at each heldout cosmology; the slope interval describes uncertainty across the
heldout cosmologies themselves.


## Paper-ready verification figures

This cell exports the conditional-recovery result and its comparison figures at the exact paper width.
The conditional figure places recovered-versus-requested $\Omega_m$ on the left panel and full empirical
coverage curves on the right panel for $N_{2D}=2^7,2^{10},2^{14}$. Both panels use the same individual
generated-map probe recoveries over the 32 held-out cosmologies. The left-panel points are medians with
16th--84th percentile error bars; the right-panel markers identify the nominal 68\% and 95\% intervals. It also
saves the frozen VGG16+MLP heldout-real slope/$R^2$ summary used to establish which conditional directions
the probe can verify. The U-Net-128 audit uses $N_{2D}=2^6,2^8,2^{10},2^{12},2^{15}$
to show copying, the intermediate degradation, and recovery at high data. The $\mathrm{FD}_{\mathrm{SSCD}}$
annotation compares 512 generated maps with 512 held-out real maps and divides by the Fréchet distance between
two disjoint 512-map halves of the held-out real set. The compact CSV records this normalized distance and its
raw real-real baseline. This held-out-real
reference tests whether generated maps remain in distribution. By contrast, the third-row Nyquist-limited
power spectra compare the generated mean with the per-field distribution from each model's exact configured
training subset. That row tests whether the model reproduces the statistics of the distribution on which it
was trained. Its Fourier coordinate is converted to physical units with the CAMELS map width
$L=25\,h^{-1}\mathrm{Mpc}$.


In [ ]:
from pathlib import Path
import importlib
import sys
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, Markdown, display

PAPER_PROJECT_DIR = Path.cwd().resolve()
while PAPER_PROJECT_DIR != PAPER_PROJECT_DIR.parent and not (PAPER_PROJECT_DIR / '.git').exists():
    PAPER_PROJECT_DIR = PAPER_PROJECT_DIR.parent
if not (PAPER_PROJECT_DIR / '.git').exists():
    raise RuntimeError(f'Could not locate repository root from {Path.cwd().resolve()}')

paper_scripts = PAPER_PROJECT_DIR / 'scripts'
if str(paper_scripts) not in sys.path:
    sys.path.insert(0, str(paper_scripts))

import plot_nf_conditional_bias_paper_figures as paper_plotting

paper_plotting = importlib.reload(paper_plotting)
build_conditional_coverage_figure = paper_plotting.build_conditional_coverage_figure
build_generalization_figure = paper_plotting.build_generalization_figure
build_nearest_training_panels = paper_plotting.build_nearest_training_panels
build_probe_summary_figure = paper_plotting.build_probe_summary_figure
export_nearest_training_outputs = paper_plotting.export_nearest_training_outputs
save_figure = paper_plotting.save_figure

paper_figure_dir = PAPER_PROJECT_DIR / 'paper' / 'ai4science_verification' / 'figures'
paper_figure_dir.mkdir(parents=True, exist_ok=True)
full_sweep_dir = (
    PAPER_PROJECT_DIR / 'results' / 'nf_conditional_bias_fresh_full_sweep_200k'
    / 'calibration_vgg'
)
paper_inputs = {
    'samples': full_sweep_dir / 'bias_probe_per_sample_predictions.csv',
    'generalization': (
        PAPER_PROJECT_DIR / 'results' / 'nf_generalize_fig2' / 'tables'
        / 'nf_generalize_fig2_pca_full_nn_metrics.csv'
    ),
    'nearest_manifest': (
        PAPER_PROJECT_DIR / 'local' / 'nf_generalize_fig2' / 'manifest.json'
    ),
    'sscd_cache': (
        PAPER_PROJECT_DIR / 'results' / 'nf_generalize_fig2' / 'cache'
        / 'sscd_full_nn'
    ),
    'probe': (
        PAPER_PROJECT_DIR / 'results' / 'nf_conditional_bias_probe' / 'encoder'
        / 'vgg_real_probe_slope_r2_summary.csv'
    ),
}
paper_file_inputs = {
    key: path for key, path in paper_inputs.items() if key != 'sscd_cache'
}
missing_paper_inputs = [str(path) for path in paper_file_inputs.values() if not path.is_file()]
if not paper_inputs['sscd_cache'].is_dir():
    missing_paper_inputs.append(str(paper_inputs['sscd_cache']))
if missing_paper_inputs:
    raise FileNotFoundError('Missing exact paper inputs:\n' + '\n'.join(missing_paper_inputs))

paper_outputs = {
    'conditional': paper_figure_dir / 'conditional_recovery_transition.pdf',
    'conditional_coverage_table': (
        paper_figure_dir / 'conditional_recovery_coverage_curves.csv'
    ),
    'generalization': paper_figure_dir / 'generalization_transition.pdf',
    'nearest': paper_figure_dir / 'nearest_training_u128.pdf',
    'nearest_preview': paper_figure_dir / 'nearest_training_u128_preview.png',
    'nearest_table': paper_figure_dir / 'nearest_training_u128.csv',
    'nearest_caption': paper_figure_dir / 'nearest_training_u128_caption.tex',
    'probe': paper_figure_dir / 'vgg_probe_heldout_real.pdf',
}

paper_samples = pd.read_csv(paper_inputs['samples'])
paper_generalization = pd.read_csv(paper_inputs['generalization'])
conditional_figure, conditional_coverage_report = build_conditional_coverage_figure(
    paper_samples,
    bootstrap=2000,
    seed=123,
)
paper_dimensions = {
    'conditional': save_figure(conditional_figure, paper_outputs['conditional'])
}
conditional_coverage_report.to_csv(
    paper_outputs['conditional_coverage_table'], index=False
)
display(Markdown('### Conditional recovery coverage'))
display(conditional_figure)
plt.close(conditional_figure)

generalization_figure = build_generalization_figure(paper_generalization)
paper_dimensions['generalization'] = save_figure(
    generalization_figure,
    paper_outputs['generalization'],
)
display(Markdown('### Memorization-to-novelty transition'))
display(generalization_figure)
plt.close(generalization_figure)

nearest_panels = build_nearest_training_panels(
    PAPER_PROJECT_DIR,
    paper_inputs['nearest_manifest'],
    paper_inputs['sscd_cache'],
    seed=123,
    sample_label='dpm50',
)
paper_dimensions['nearest'], nearest_training_report = export_nearest_training_outputs(
    nearest_panels,
    paper_outputs['nearest'],
    paper_outputs['nearest_table'],
    preview_path=paper_outputs['nearest_preview'],
    caption_path=paper_outputs['nearest_caption'],
)
display(Markdown('### Generated samples, nearest training slices, and in-distribution check'))
display(Image(filename=str(paper_outputs['nearest_preview']), width=1100))
display(nearest_training_report)

probe_figure = build_probe_summary_figure(pd.read_csv(paper_inputs['probe']))
paper_dimensions['probe'] = save_figure(probe_figure, paper_outputs['probe'])
display(Markdown('### Frozen VGG16+MLP heldout-real validation'))
display(probe_figure)
plt.close(probe_figure)

display(Markdown('### Saved paper figures'))
for paper_name in ('conditional', 'generalization', 'nearest', 'probe'):
    paper_path = paper_outputs[paper_name]
    width, height = paper_dimensions[paper_name]
    print(f'{paper_name}: {paper_path} ({width:.3f} x {height:.3f} in)')
print(f"nearest_table: {paper_outputs['nearest_table']} ({len(nearest_training_report)} rows)")
print(f"nearest_caption: {paper_outputs['nearest_caption']}")
print(f"nearest_preview: {paper_outputs['nearest_preview']} (300 dpi)")
print(
    f"conditional_coverage_table: {paper_outputs['conditional_coverage_table']} "
    f"({len(conditional_coverage_report)} rows)"
)

display(Markdown('### Exact saved $\\Omega_m$ coverage'))
display(
    conditional_coverage_report[
        conditional_coverage_report['plotted']
        & conditional_coverage_report['nominal_coverage'].isin([0.68, 0.95])
    ][
        [
            'dataset_size', 'nominal_coverage', 'empirical_coverage',
            'coverage_ci16', 'coverage_ci84', 'n_heldout',
            'draws_per_cosmology',
        ]
    ].sort_values(['dataset_size', 'nominal_coverage']).reset_index(drop=True)
)


The PDFs above contain no figure-level titles; put the scientific description, ideal-calibration diagonal,
and training protocol in the LaTeX captions. In the conditional figure, the left panel shows the measured
recovered-versus-requested $\Omega_m$ relation and the right panel shows coverage at
$N_{2D}=2^7,2^{10},2^{14}$; curves below the diagonal are overconfident.
The U-Net-128 table records the copying similarity, normalized SSCD Fr\'echet distance and real-real
baseline, matched evaluation counts, and exact-subset configuration for every displayed column. The generated
caption file states explicitly that $\mathrm{FD}_{\mathrm{SSCD}}$ uses held-out real fields, whereas the
power-spectrum distribution uses each model's exact training subset.
